# SENSERO — Spatial leakage enforcement (train vs eval, same caption)

Runs **directly after** `primary_split_mgrs.ipynb`. Single-pass enforcement
that eliminates spatial leakage between train and (val ∪ test) patches
sharing the same CLC codes (hence the same caption).

**Replaces** the old two-step pipeline (`spatial_buffer_enforcement.ipynb` +
separate train↔eval notebook). The val↔val / val↔test / test↔test buffer
is **intentionally dropped** — only train vs evaluation separation matters.

## Problem

Dense extraction (NN ≈ 3.9 km, patch = 3.36 km) means nearly every eval
patch has adjacent train patches. When they share the same CLC codes the
model can exploit spatial autocorrelation — the train patch is essentially
a near-duplicate of the evaluation target.

## Strategy

For each val/test patch V that has a same-CLC train neighbour within
`BUFFER_M`:

1. Find a train patch S with the **same CLC codes** that has **no**
   same-CLC train neighbours within `BUFFER_M` (i.e. S would be a
   "clean" eval patch after promotion)
2. **Swap:** V → train, S → val/test (inherits V's original split label)
3. If no valid S exists → log and accept (ultra-rare CLC combos)

## Guarantees

- **Counts preserved:** 7 000 / 1 500 / 1 500 exactly
- **CLC distribution preserved:** every swap is same-CLC ↔ same-CLC
- **Residual leakage:** ~8 pairs from rare combos (521, 411+321 …),
  documented in report


In [11]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree
from collections import defaultdict

# ──────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────
ROOT       = "/home/ubuntu/SENSERO/GeoTiff/"
GPKG_PATH  = "/home/ubuntu/SENSERO/sensero_patches.gpkg"
METRIC_CRS = "EPSG:3035"
BUFFER_M   = 5000
RANDOM_STATE = 42

# Input: canonical split from primary_split_mgrs.ipynb
CANONICAL_IN = os.path.join(ROOT, "canonical_checkpoint.parquet")

# Outputs
SPLIT_OUT_CSV     = os.path.join(ROOT, "split_summary.csv")
SPLIT_OUT_PARQUET = os.path.join(ROOT, "split_summary.parquet")
SWAP_LOG_FILE      = os.path.join(ROOT, "leakage_swap_log.csv")
RESIDUAL_LOG_FILE  = os.path.join(ROOT, "leakage_residual.csv")
FAILURE_LOG_FILE   = os.path.join(ROOT, "leakage_failures.csv")

# Target counts
TARGET_TRAIN = 7000
TARGET_VAL   = 1500
TARGET_TEST  = 1500

# All patch sizes to propagate
ALL_SIZES = [64, 112, 120, 128, 224, 256, 280, 336]

RNG = np.random.default_rng(RANDOM_STATE)

print(f"Buffer : {BUFFER_M} m")
print(f"CRS    : {METRIC_CRS}")
print(f"Input  : {CANONICAL_IN}")


Buffer : 5000 m
CRS    : EPSG:3035
Input  : /home/ubuntu/SENSERO/GeoTiff/canonical_checkpoint.parquet


In [12]:
# ──────────────────────────────────────────────
# LOAD
# ──────────────────────────────────────────────

# Canonical split
df_canon = pd.read_parquet(CANONICAL_IN)
print(f"Canonical split: {len(df_canon)} patches")
print(df_canon["split"].value_counts())

counts = df_canon["split"].value_counts().to_dict()
assert counts.get("train") == TARGET_TRAIN, f"train={counts.get('train')} != {TARGET_TRAIN}"
assert counts.get("val")   == TARGET_VAL,   f"val={counts.get('val')} != {TARGET_VAL}"
assert counts.get("test")  == TARGET_TEST,  f"test={counts.get('test')} != {TARGET_TEST}"
print("✓ Input counts match targets")

# GeoPackage
gdf = gpd.read_file(GPKG_PATH)
gdf["BaseFilename"] = gdf["filename"].str.removesuffix(".tif")
gdf_m = gdf[["BaseFilename", "geometry"]].to_crs(METRIC_CRS).copy()
gdf_m["centroid"] = gdf_m.geometry.centroid
print(f"\nGeoPackage: {len(gdf_m)} features → {METRIC_CRS}")

# CLC codes from any metadata parquet
for sz in ALL_SIZES:
    meta_path = os.path.join(ROOT, f"Patch_{sz}", f"metadata_{sz}.parquet")
    if os.path.exists(meta_path):
        df_meta = pd.read_parquet(meta_path, columns=["BaseFilename", "CLC_codes"])
        print(f"CLC codes from: metadata_{sz}.parquet")
        break
else:
    raise FileNotFoundError("No metadata parquet found for CLC codes")

# Merge all
gdf_m = gdf_m.merge(df_canon[["BaseFilename", "split"]], on="BaseFilename", how="inner")
gdf_m = gdf_m.merge(df_meta, on="BaseFilename", how="inner")

n_missing = len(df_canon) - len(gdf_m)
if n_missing:
    raise ValueError(f"{n_missing} canonical patches missing from GeoPackage!")
print(f"\nMerged: {len(gdf_m)} patches with geometry + split + CLC_codes")


Canonical split: 10000 patches
split
train    7000
val      1500
test     1500
Name: count, dtype: int64
✓ Input counts match targets

GeoPackage: 10000 features → EPSG:3035
CLC codes from: metadata_280.parquet

Merged: 10000 patches with geometry + split + CLC_codes


In [13]:
# ──────────────────────────────────────────────
# SPATIAL INDEX
# ──────────────────────────────────────────────
coords    = np.array([(c.x, c.y) for c in gdf_m["centroid"]])
tree      = cKDTree(coords)
filenames = gdf_m["BaseFilename"].values
splits    = gdf_m["split"].values.copy()
clc       = gdf_m["CLC_codes"].values

eval_idx  = set(np.where(np.isin(splits, ["val", "test"]))[0])
train_idx = set(np.where(splits == "train")[0])

print(f"KD-tree: {len(coords)} points")
print(f"Train: {len(train_idx)}  |  Eval: {len(eval_idx)}")


KD-tree: 10000 points
Train: 7000  |  Eval: 3000


In [14]:
# ──────────────────────────────────────────────
# DETECT LEAKAGE
# ──────────────────────────────────────────────
affected_eval = {}
all_leaky_pairs = []

for ei in eval_idx:
    neighbours = tree.query_ball_point(coords[ei], BUFFER_M)
    leaky = [ni for ni in neighbours if ni in train_idx and clc[ni] == clc[ei]]
    if leaky:
        affected_eval[ei] = leaky
        for ni in leaky:
            all_leaky_pairs.append({
                "eval_file":  filenames[ei],
                "eval_split": splits[ei],
                "train_file": filenames[ni],
                "CLC_codes":  clc[ei],
                "distance_m": np.linalg.norm(coords[ei] - coords[ni]),
            })

print(f"Leaky pairs:        {len(all_leaky_pairs)}")
print(f"Affected eval:      {len(affected_eval)} / {len(eval_idx)}")
print(f"  val:  {sum(1 for ei in affected_eval if splits[ei] == 'val')}")
print(f"  test: {sum(1 for ei in affected_eval if splits[ei] == 'test')}")

train_by_clc = defaultdict(list)
for ti in train_idx:
    train_by_clc[clc[ti]].append(ti)
print(f"\nTrain CLC groups: {len(train_by_clc)}")

if len(affected_eval) == 0:
    print("\n✓ No leakage detected — nothing to do.")


Leaky pairs:        300
Affected eval:      224 / 3000
  val:  126
  test: 98

Train CLC groups: 3695


In [15]:
# ──────────────────────────────────────────────
# SWAP ENGINE
# ──────────────────────────────────────────────
# For each affected eval patch, find a train patch with same CLC that
# would be "clean" as an eval patch (no same-CLC train neighbours).
#
# Sort hardest-first (fewest candidates) so constrained patches get
# first pick from the full pool.

current_eval  = set(eval_idx)
current_train = set(train_idx)
used = set()

swap_log = []
failures = []

affected_sorted = sorted(
    affected_eval.keys(),
    key=lambda ei: len(train_by_clc.get(clc[ei], []))
)

for ei in affected_sorted:
    if ei not in current_eval:
        continue

    clc_code = clc[ei]
    candidates = list(train_by_clc.get(clc_code, []))
    RNG.shuffle(candidates)

    found = False
    for ci in candidates:
        if ci in used:
            continue

        # After swap: ci → eval, ei → train
        # ci must have NO same-CLC train neighbours
        near = tree.query_ball_point(coords[ci], BUFFER_M)
        effective_train = (current_train - {ci}) | {ei}
        if any(ni in effective_train and clc[ni] == clc_code for ni in near):
            continue

        # Valid swap
        swap_log.append({
            "eval_demoted":    filenames[ei],
            "original_split":  splits[ei],
            "train_promoted":  filenames[ci],
            "CLC_codes":       clc_code,
        })
        current_eval.discard(ei)
        current_eval.add(ci)
        current_train.discard(ci)
        current_train.add(ei)
        used.add(ci)
        found = True
        break

    if not found:
        failures.append({
            "eval_file":    filenames[ei],
            "eval_split":   splits[ei],
            "CLC_codes":    clc_code,
            "n_candidates": len(candidates),
            "reason":       "no safe same-CLC replacement in train",
        })

print(f"Swaps completed: {len(swap_log)}")
print(f"Failures:        {len(failures)}")
if affected_eval:
    print(f"Success rate:    {100 * len(swap_log) / len(affected_eval):.1f}%")


Swaps completed: 214
Failures:        10
Success rate:    95.5%


In [16]:
# ──────────────────────────────────────────────
# APPLY NEW SPLIT LABELS
# ──────────────────────────────────────────────
new_splits = splits.copy()

# Build lookup: promoted train filename → inherited split label
promoted_label = {}
for entry in swap_log:
    promoted_label[entry["train_promoted"]] = entry["original_split"]

for i in range(len(new_splits)):
    fn = filenames[i]
    if i in current_eval and fn in promoted_label:
        new_splits[i] = promoted_label[fn]
    elif i in current_train and new_splits[i] != "train":
        # This was a demoted eval patch → now train
        new_splits[i] = "train"

counts = pd.Series(new_splits).value_counts()
print("Split distribution after enforcement:")
print(counts.sort_index())

assert counts.get("train", 0) == TARGET_TRAIN, \
    f"train={counts.get('train', 0)} != {TARGET_TRAIN}"
assert counts.get("val", 0) == TARGET_VAL, \
    f"val={counts.get('val', 0)} != {TARGET_VAL}"
assert counts.get("test", 0) == TARGET_TEST, \
    f"test={counts.get('test', 0)} != {TARGET_TEST}"
print("\n✓ Counts preserved: 7000 / 1500 / 1500")


Split distribution after enforcement:
test     1500
train    7000
val      1500
Name: count, dtype: int64

✓ Counts preserved: 7000 / 1500 / 1500


In [17]:
# ──────────────────────────────────────────────
# VERIFY: residual leakage scan
# ──────────────────────────────────────────────
final_eval  = set(np.where(np.isin(new_splits, ["val", "test"]))[0])
final_train = set(np.where(new_splits == "train")[0])

residual = []
for ei in final_eval:
    for ni in tree.query_ball_point(coords[ei], BUFFER_M):
        if ni in final_train and clc[ni] == clc[ei]:
            residual.append({
                "eval_file":  filenames[ei],
                "eval_split": new_splits[ei],
                "train_file": filenames[ni],
                "CLC_codes":  clc[ei],
                "distance_m": np.linalg.norm(coords[ei] - coords[ni]),
            })

df_residual = pd.DataFrame(residual) if residual else pd.DataFrame()

print(f"Residual same-CLC eval↔train pairs: {len(residual)}")
if len(residual) == 0:
    print("✓ ZERO same-caption spatial leakage")
else:
    print(f"  Unique CLC codes:   {df_residual['CLC_codes'].nunique()}")
    print(f"  Unique eval patches: {df_residual['eval_file'].nunique()}")
    print(f"  Unique train patches: {df_residual['train_file'].nunique()}")
    print()
    print("Breakdown:")
    print(df_residual["CLC_codes"].value_counts().to_string())
    print()
    print("Ultra-rare CLC combos with no valid swap — accepted and documented.")


Residual same-CLC eval↔train pairs: 46
  Unique CLC codes:   11
  Unique eval patches: 40
  Unique train patches: 41

Breakdown:
CLC_codes
211              29
521               7
311               2
321, 211, 311     1
411, 512          1
211, 231          1
211, 231, 112     1
311, 211, 112     1
311, 231          1
312, 231          1
231, 312, 324     1

Ultra-rare CLC combos with no valid swap — accepted and documented.


In [18]:
# ──────────────────────────────────────────────
# SAVE
# ──────────────────────────────────────────────
master_new = pd.DataFrame({
    "BaseFilename": filenames,
    "split": new_splits,
})

master_new.to_csv(SPLIT_OUT_CSV, index=False)
master_new.to_parquet(SPLIT_OUT_PARQUET, index=False)
print(f"[SAVE] {SPLIT_OUT_PARQUET}")
print(f"       {master_new['split'].value_counts().sort_index().to_dict()}")

if swap_log:
    pd.DataFrame(swap_log).to_csv(SWAP_LOG_FILE, index=False)
    print(f"[SAVE] {SWAP_LOG_FILE} ({len(swap_log)} swaps)")

if residual:
    df_residual.to_csv(RESIDUAL_LOG_FILE, index=False)
    print(f"[SAVE] {RESIDUAL_LOG_FILE} ({len(residual)} pairs)")

if failures:
    pd.DataFrame(failures).to_csv(FAILURE_LOG_FILE, index=False)
    print(f"[SAVE] {FAILURE_LOG_FILE} ({len(failures)} entries)")


[SAVE] /home/ubuntu/SENSERO/GeoTiff/split_summary.parquet
       {'test': 1500, 'train': 7000, 'val': 1500}
[SAVE] /home/ubuntu/SENSERO/GeoTiff/leakage_swap_log.csv (214 swaps)
[SAVE] /home/ubuntu/SENSERO/GeoTiff/leakage_residual.csv (46 pairs)
[SAVE] /home/ubuntu/SENSERO/GeoTiff/leakage_failures.csv (10 entries)


In [19]:
# ──────────────────────────────────────────────
# PROPAGATE to all patch sizes
# ──────────────────────────────────────────────
# Split is per-location, identical for all patch sizes.
# Just write the master split into each Patch_<size>/ folder.

print("Writing split_summary to all patch-size folders...\n")

for size in ALL_SIZES:
    patch_dir = os.path.join(ROOT, f"Patch_{size}")
    if not os.path.isdir(patch_dir):
        print(f"[SKIP] {size}px: folder does not exist")
        continue

    out_pq  = os.path.join(patch_dir, "split_summary.parquet")
    out_csv = os.path.join(patch_dir, "split_summary.csv")
    master_new.to_parquet(out_pq, index=False)
    master_new.to_csv(out_csv, index=False)
    print(f"[SAVE] {size}px: {len(master_new)} rows → split_summary.*")

print("\nDONE.")


Writing split_summary to all patch-size folders...

[SAVE] 64px: 10000 rows → split_summary.*
[SAVE] 112px: 10000 rows → split_summary.*
[SAVE] 120px: 10000 rows → split_summary.*
[SAVE] 128px: 10000 rows → split_summary.*
[SAVE] 224px: 10000 rows → split_summary.*
[SAVE] 256px: 10000 rows → split_summary.*
[SAVE] 280px: 10000 rows → split_summary.*
[SAVE] 336px: 10000 rows → split_summary.*

DONE.


In [21]:
# ──────────────────────────────────────────────
# SUMMARY
# ──────────────────────────────────────────────
print("=" * 62)
print("  SPATIAL LEAKAGE ENFORCEMENT — FINAL REPORT")
print("=" * 62)
print()

counts = master_new["split"].value_counts().sort_index()
for s, c in counts.items():
    bar = "█" * int(c / 200)
    print(f"  {s:8s}  {c:5d}  {bar}")
print(f"  {'TOTAL':8s}  {len(master_new):5d}")

print()
print(f"  Leaky pairs (before):  {len(all_leaky_pairs)}")
print(f"  Swaps performed:       {len(swap_log)}")
print(f"  Swap failures:         {len(failures)}")
print(f"  Residual pairs:        {len(residual)}")
print()
print(f"  ✓ Counts:  7000 / 1500 / 1500")
print(f"  ✓ CLC distribution: preserved (same-CLC swaps only)")

if len(residual) == 0:
    print(f"  ✓ Leakage: ZERO")
else:
    print(f"  ⚠ Leakage: {len(residual)} residual pairs (rare CLC, documented)")

print()
print("  Pipeline: primary_split_mgrs → THIS → merge_metadata")
print("=" * 62)


  SPATIAL LEAKAGE ENFORCEMENT — FINAL REPORT

  test       1500  ███████
  train      7000  ███████████████████████████████████
  val        1500  ███████
  TOTAL     10000

  Leaky pairs (before):  300
  Swaps performed:       214
  Swap failures:         10
  Residual pairs:        46

  ✓ Counts:  7000 / 1500 / 1500
  ✓ CLC distribution: preserved (same-CLC swaps only)
  ⚠ Leakage: 46 residual pairs (rare CLC, documented)

  Pipeline: primary_split_mgrs → THIS → merge_metadata
